# FerroSim — Programmatic GA–DKL: discovering experimental operation sequences

**Self-contained, standalone, editable notebook.** The optimized object is a variable-length program of six physical primitives: **ramp, hold, reverse, pulse, damped oscillatory ringdown, and oscillatory burst**. The compiler produces continuous normalized `E/Ec` waveforms, with zero-field/zero-slope at the start and end. The *program length, primitive types, durations and primitive parameters* can mutate.

**Controlled scientific ablation:** `simple` grammar (ramp, hold, reverse) vs `full` grammar (all six primitives), independently searched using GA–DKL and QD–DKL. Identical shared initial protocols and disorder configurations; the broader grammar is not assumed better. Methods here are exactly **two** genetic-programming variants with a waveform DKL surrogate; this notebook does **not** claim to implement a trained GFlowNet or generative diffusion model.

**Execution modes:** By default, run-all performs source installation, compiler tests, and waveform visualization **without costly physical oracle calls**. Set `RUN_EXPERIMENT=True` and `PILOT=True` for a small *actual FerroSim* study; set `PILOT=False` for the full **20 maps × 3 seeds × 2 grammars × 2 methods × 1,088 evaluations = 261,120** physical-oracle evaluations. The full RK4 experiment is potentially very computationally expensive and results are **not precomputed**. The initial pilot uses first-order legacy Euler, *not* comparable quantitatively to RK4 Fourier or Legendre results.

No GitHub integration or uploads.

## 1. Environment and run settings

In [ ]:
import os,sys,pathlib,subprocess,base64,zlib,io,importlib
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
ROOT=Path.cwd()/'FerroSim_Grammar_runtime';ROOT.mkdir(exist_ok=True)
RUN_EXPERIMENT=False    # Set True to evaluate the FerroSim physical oracle
PILOT=True             # True: 1 map, 1 seed, 12 eval/run; False: 20 maps, 3 seeds, 1088 eval/run
RUN_TRANSFER=False     # Extra physical evaluations, only sensible for multi-map results
RUN_REFINEMENT=False   # Higher-fidelity validation of champion controls
os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
os.environ.setdefault('OMP_NUM_THREADS','1')
for package in ('numpy','pandas','matplotlib','scipy','numba','torch'):
    try:importlib.import_module(package)
    except ImportError:subprocess.check_call([sys.executable,'-m','pip','install',package])
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
print('Self-contained runtime:',ROOT)

## 2. Complete editable source modules (embedded; zero external archives)

In [ ]:
%%writefile FerroSim_Grammar_runtime/representation.py
"""Variable-length procedural control grammar; each primitive has an explicit physical meaning.
Tokens are (operation ID, relative duration, target, amplitude, frequency/cycles, decay time).
A fixed-width encoding is storage only: inactive tokens are ignored by the compiler.
"""
import numpy as np
from scipy.signal import find_peaks
OPS=('ramp','hold','reverse','pulse','ringdown','burst')
RAMP,HOLD,REVERSE,PULSE,RINGDOWN,BURST=range(6)
MIN_K,MAX_K=2,12
STRIDE=6
DIM=1+MAX_K*STRIDE
DRIVE_SAMPLES=900

def encode(ops):
    k=len(ops);assert MIN_K<=k<=MAX_K
    z=np.zeros(DIM);z[0]=k
    z[1:1+STRIDE*k]=np.asarray(ops,float).reshape(-1)
    return z

def decode(z):
    z=np.asarray(z,float).ravel();k=int(np.clip(round(z[0]),MIN_K,MAX_K))
    ops=z[1:1+STRIDE*k].reshape(k,STRIDE).copy()
    ops[:,0]=np.clip(np.rint(ops[:,0]),0,5)
    ops[:,1]=np.clip(ops[:,1],.03,5)
    ops[:,2:4]=np.clip(ops[:,2:4],-70,70)
    ops[:,4]=np.clip(ops[:,4],.1,12)
    ops[:,5]=np.clip(ops[:,5],.08,5)
    return ops

def compile_one(z,n=900,return_meta=False):
    seq=decode(z);u=np.linspace(0,1,n);wave=np.zeros(n)
    d=np.clip(seq[:,1],.03,5);d=d/d.sum()*.9
    bounds=.05+np.r_[0,np.cumsum(d)];bounds[-1]=.95
    level=0.
    for i,(op,_,target,amp,freq,tau) in enumerate(seq):
        left,right=bounds[i:i+2]
        mask=(u>=left)&(u<=right if i==len(seq)-1 else u<right)
        q=np.clip((u[mask]-left)/(right-left),0,1)
        op=int(op)
        if op==RAMP:
            f=level+(target-level)*q;newlevel=target
        elif op==REVERSE:
            newlevel=-np.sign(level if abs(level)>.01 else (target if abs(target)>.01 else 1))*abs(target)
            f=level+(newlevel-level)*q
        elif op==HOLD:
            f=np.full(q.shape,level);newlevel=level
        elif op==PULSE:
            f=level+amp*np.sin(np.pi*q)**2;newlevel=level
        elif op==RINGDOWN:
            # Oscillatory ringdown gated smoothly to ensure continuity at both ends.
            envelope=np.sin(np.pi*q)**2*np.exp(-q/tau)
            f=level+amp*envelope*np.sin(2*np.pi*freq*q);newlevel=level
        else:
            f=level+amp*np.sin(np.pi*q)**2*np.sin(2*np.pi*freq*q);newlevel=level
        wave[mask]=f;level=float(newlevel)
    # Closing ramp: fixed 0.025 duration, then zero field and slope at end.
    close=(u>.95)&(u<.975);wave[close]=level*(1-(u[close]-.95)/.025)
    if return_meta:return u,wave,bounds
    return wave

def fields(x,n=900):
    z=np.atleast_2d(np.asarray(x,float));return np.stack([compile_one(y,n) for y in z])

def complexity(x,n=1000):
    z=np.atleast_2d(x);w=fields(z,n)
    result={k:np.zeros(len(z),dtype=int) for k in ['segments','effective_segments','turns','raw_turns','turns_1pct','turns_5pct','n_ramps','n_pulses','n_ringdowns','n_bursts','n_holds','n_reverses','parameter_count']}
    for i,x in enumerate(z):
        ops=decode(x);k=len(ops);result['segments'][i]=k;result['effective_segments'][i]=k
        for label,idx in [('n_ramps',RAMP),('n_pulses',PULSE),('n_ringdowns',RINGDOWN),('n_bursts',BURST),('n_holds',HOLD),('n_reverses',REVERSE)]:result[label][i]=np.sum(ops[:,0]==idx)
        result['parameter_count'][i]=2*k+sum(int(o in [PULSE,RINGDOWN,BURST])*(1+int(o in [RINGDOWN,BURST])*2) for o in ops[:,0])
        peak=np.ptp(w[i]);
        for key,prom in [('raw_turns',0),('turns_1pct',.01),('turns',.02),('turns_5pct',.05)]:
            result[key][i]=int(len(find_peaks(w[i],prominence=peak*prom)[0])+len(find_peaks(-w[i],prominence=peak*prom)[0])) if peak else 0
    result['peak_ec']=np.max(np.abs(w),axis=1);result['rms_ec']=np.sqrt(np.mean(w*w,axis=1))
    return result

def random_op(rng,grammar='full'):
    available=[RAMP,HOLD,REVERSE] if grammar=='simple' else list(range(6))
    op=int(rng.choice(available));return np.array([op,float(rng.uniform(.25,1.6)),float(rng.uniform(-40,40)),float(rng.uniform(-35,35)),float(rng.uniform(.5,7)),float(rng.uniform(.15,1.7))])

def sample_one(rng,grammar='full',k=None):
    k=int(rng.integers(MIN_K,MAX_K+1)) if k is None else int(k)
    ops=[random_op(rng,grammar) for _ in range(k)];ops[0][0]=RAMP
    return encode(ops)

def initial_design(seed,n=8,grammar='full'):
    rng=np.random.default_rng(seed);ans=[]
    # shared initial candidates across both grammars, for defensible ablation
    for peak in (10.,-10.,25.,-25.,40.,-40.):
        ops=np.array([[RAMP,.9,peak,0,2,1],[HOLD,.3,0,0,2,1],[RAMP,.9,0,0,2,1]],float)
        ans.append(encode(ops))
        if len(ans)>=n:return np.array(ans)
    while len(ans)<n:ans.append(sample_one(rng,grammar))
    return np.asarray(ans)

def mutate(z,rng,grammar='full',force=None):
    ops=decode(z);k=len(ops)
    choices=['insert','delete','replace','duration','target','amplitude','frequency','decay','swap','rescale']
    op=force if force else str(rng.choice(choices,p=[.13,.13,.14,.14,.14,.12,.06,.05,.04,.05]))
    j=int(rng.integers(k))
    if op=='insert' and k<MAX_K:
        ops=np.insert(ops,j,random_op(rng,grammar),axis=0)
    elif op=='delete' and k>MIN_K:
        ops=np.delete(ops,j,axis=0)
    elif op=='replace':ops[j]=random_op(rng,grammar)
    elif op=='duration':ops[j,1]*=np.exp(rng.normal(0,.45))
    elif op=='target':ops[j,2]+=rng.normal(0,10)
    elif op=='amplitude':ops[j,3]+=rng.normal(0,10)
    elif op=='frequency':ops[j,4]*=np.exp(rng.normal(0,.3))
    elif op=='decay':ops[j,5]*=np.exp(rng.normal(0,.4))
    elif op=='swap' and k>2:ops[[j,(j+1)%k]]=ops[[(j+1)%k,j]]
    elif op=='rescale':ops[:,2:4]*=np.exp(rng.normal(0,.3))
    if grammar=='simple':
        for row in ops:
            if int(row[0])>REVERSE:row[0]=int(rng.choice([RAMP,HOLD,REVERSE]))
    ops[:,1]=np.clip(ops[:,1],.03,5)
    ops[:,2:4]=np.clip(ops[:,2:4],-70,70)
    ops[:,4]=np.clip(ops[:,4],.1,12);ops[:,5]=np.clip(ops[:,5],.08,5)
    return encode(ops),op


In [ ]:
%%writefile FerroSim_Grammar_runtime/oracle.py
"""Accelerated deterministic FerroSim GLD evaluator.

Physics: native rama_dev_updated@5eb258e, gamma=1, tetragonal, k=100,
dep_alpha=.3, finite-grid interactions with fixed /4 neighbor normalization.
Default integration is subdivided RK4, NOT the native first-order Euler method.
No field-amplitude bound or state clipping is imposed in RK4. The native
calDeriv clipping guard |P|=10 is explicitly diagnosed if ever crossed.
Native periodic central-difference curl and an interior diagnostic are returned.
"""
from __future__ import annotations
import os
import math
from pathlib import Path
import numpy as np
os.environ.setdefault("NUMBA_NUM_THREADS", "2")
from numba import njit, prange

N = 20
ALPHA1, ALPHA2, ALPHA3 = -1.6, 12.2, 40.0
COUPLING, DEP_ALPHA = 100.0, 0.3
EC = (-2.0 / 3.0) * ALPHA1 * np.sqrt(abs(-ALPHA1/(3.0*ALPHA2)))
TIME = np.linspace(0.0, 4.0, 1200)[:950]
DT = float(TIME[1]-TIME[0])
SOURCE_COMMIT = "5eb258e2a47d10b3d9bdb9e94dd9879b8410024c"
_LOCAL_DEFECT_PATH = Path(os.environ.get("FERROSIM_DEFECT_PATH", str(Path(__file__).with_name("defect_list_seed.npy"))))
DEFAULT_DEFECT_PATH = (_LOCAL_DEFECT_PATH if _LOCAL_DEFECT_PATH.exists()
                       else Path(__file__).resolve().parents[1]/"GA-DKL/defect_list_seed.npy")
_DEFECTS = None


def default_defects():
    global _DEFECTS
    if _DEFECTS is None:
        _DEFECTS = np.load(DEFAULT_DEFECT_PATH, allow_pickle=False).astype(np.float64).reshape(N,N,2)
    return _DEFECTS.copy()


@njit(cache=True)
def _rhs(p, ex, defects, coupling, dep_alpha, native_clip):
    n = p.shape[1]
    out = np.empty_like(p)
    mx, my = 0.0, 0.0
    for i in range(n):
        for j in range(n):
            mx += p[0,i,j]
            my += p[1,i,j]
    mx /= n*n
    my /= n*n
    for i in range(n):
        for j in range(n):
            x, y = p[0,i,j], p[1,i,j]
            if native_clip:
                x = min(10.,max(-10.,x))
                y = min(10.,max(-10.,y))
            sx, sy = 0.0, 0.0
            # Same finite-grid graph as native knn_search(r=1.1).
            if i>0:
                sx += p[0,i-1,j]; sy += p[1,i-1,j]
            if j>0:
                sx += p[0,i,j-1]; sy += p[1,i,j-1]
            if j<n-1:
                sx += p[0,i,j+1]; sy += p[1,i,j+1]
            if i<n-1:
                sx += p[0,i+1,j]; sy += p[1,i+1,j]
            out[0,i,j] = -(2*ALPHA1*x + 4*ALPHA2*x*x*x + 2*ALPHA3*y*y*x
                           + coupling*(x-sx/4.) - ex + dep_alpha*mx - defects[i,j,0])
            out[1,i,j] = -(2*ALPHA1*y + 4*ALPHA2*y*y*y + 2*ALPHA3*x*x*y
                           + coupling*(y-sy/4.) + dep_alpha*my - defects[i,j,1])
    return out


@njit(cache=True)
def _maxabs(p):
    q = 0.0
    for k in range(p.size):
        v = abs(p.flat[k])
        if not np.isfinite(v):
            return np.inf
        if v>q:
            q=v
    return q


@njit(cache=True)
def _curl(p):
    n=p.shape[1]
    c=np.empty((n,n),dtype=np.float64)
    total, interior = 0.,0.
    for i in range(n):
        for j in range(n):
            v=.5*(p[1,(i+1)%n,j]-p[1,(i-1)%n,j]-p[0,i,(j+1)%n]+p[0,i,(j-1)%n])
            c[i,j]=v
            total+=abs(v)
            if 0<i<n-1 and 0<j<n-1:
                interior+=abs(v)
    return total,interior,c


@njit(cache=True)
def _integrate(field, defects, initial, fidelity, method, history, coupling, dep_alpha, max_substeps):
    """method 0: native-style Euler, method 1: curvature-subdivided RK4."""
    p=initial.copy()
    steps=field.size
    n=p.shape[1]
    hst=np.full((2,steps,n,n),np.nan,dtype=np.float64) if history else np.empty((2,1,n,n),dtype=np.float64)
    hst[:,0,:,:]=p
    peak=_maxabs(p)
    defects_peak=0.
    for q in range(defects.size):
        defects_peak=max(defects_peak,abs(defects.flat[q]))
    rhs_calls=0
    subdivisions=0
    guard_exceeded=peak>10.
    status=0
    for t in range(1,steps):
        e0,e1=field[t-1]*EC,field[t]*EC
        if method==0:
            count=fidelity
        else:
            # A conservative estimate of the local Jacobian row-sum magnitude.
            # The forcing equilibrium scale anticipates sharp field increases.
            pbound=max(_maxabs(p),((max(abs(e0),abs(e1))+defects_peak+10.)/(4*ALPHA2))**(1/3)+.3)
            curvature=(12*ALPHA2+6*abs(ALPHA3))*pbound*pbound+2*coupling+abs(2*ALPHA1)+dep_alpha
            needed=DT*curvature*fidelity
            # Check in floating point before integer conversion: finite but huge
            # requested fields can otherwise overflow the subdivision counter.
            if not np.isfinite(needed) or needed>max_substeps:
                status=2
                break
            count=max(2*fidelity,int(math.ceil(needed)))
        if count>max_substeps:
            status=2
            break
        h=DT/count
        for sub in range(count):
            f0=sub/count
            f1=(sub+1)/count
            if method==0:
                # At fidelity=1 this exactly uses field[t], as native calcODE does.
                ex=e0+(e1-e0)*f1
                p=p+h*_rhs(p,ex,defects,coupling,dep_alpha,True)
                rhs_calls+=1
            else:
                ea=e0+(e1-e0)*f0
                eb=e0+(e1-e0)*f1
                em=.5*(ea+eb)
                k1=_rhs(p,ea,defects,coupling,dep_alpha,False)
                k2=_rhs(p+.5*h*k1,em,defects,coupling,dep_alpha,False)
                k3=_rhs(p+.5*h*k2,em,defects,coupling,dep_alpha,False)
                k4=_rhs(p+h*k3,eb,defects,coupling,dep_alpha,False)
                p=p+(h/6.)*(k1+2*k2+2*k3+k4)
                rhs_calls+=4
            pm=_maxabs(p)
            if not np.isfinite(pm):
                status=1
                break
            peak=max(peak,pm)
            guard_exceeded=guard_exceeded or pm>10.
        subdivisions+=count
        if history:
            hst[:,t,:,:]=p
        if status:
            break
    if not history:
        hst[:,0,:,:]=p
    if status:
        return np.nan,np.nan,p,peak,guard_exceeded,status,rhs_calls,subdivisions,hst
    reward,interior,_=_curl(p)
    return reward,interior,p,peak,guard_exceeded,status,rhs_calls,subdivisions,hst


@njit(cache=True,parallel=True)
def _batch(fields,defects,initial,fidelity,method,coupling,dep_alpha,max_substeps):
    batch=fields.shape[0]
    n=initial.shape[1]
    reward=np.empty(batch); interior=np.empty(batch); peak=np.empty(batch)
    final=np.empty((batch,2,n,n)); guards=np.empty(batch,dtype=np.bool_)
    status=np.empty(batch,dtype=np.int64); rhs=np.empty(batch,dtype=np.int64); subdivisions=np.empty(batch,dtype=np.int64)
    for q in prange(batch):
        r,j,p,m,g,s,nrhs,nsub,_=_integrate(fields[q],defects,initial,fidelity,method,False,coupling,dep_alpha,max_substeps)
        reward[q]=r; interior[q]=j; final[q]=p; peak[q]=m
        guards[q]=g; status[q]=s; rhs[q]=nrhs; subdivisions[q]=nsub
    return reward,interior,final,peak,guards,status,rhs,subdivisions


def evaluate_fields(fields_ec, fidelity=1, return_history=False, method="rk4", defects_ec=None,
                    initial_p=None, coupling=COUPLING, dep_alpha=DEP_ALPHA, max_substeps=100000):
    """Evaluate x-only waveforms with a fresh state for every candidate.

    Parameters
    ----------
    fields_ec : (batch,900) or (batch,950), or a single corresponding vector.
        Actual native-interface field E/Ec, WITHOUT normalization or amplitude cap.
        With 900 samples, 50 repeats of the final field are appended.
    fidelity : integer >=1
        Multiplies RK4 temporal resolution. Legacy Euler uses this many substeps.
    return_history : bool
        Add P_history shape (batch,2,950,n,n), mean_P and curl_history.
    method : "rk4" (default) or "legacy_euler"
        RK4 uses linear field interpolation and no state clipping.
    initial_p : optional (n,n,2) or (2,n,n), default zeros on20x20.
    defects_ec : optional (n*n,2) or (n,n,2), default published19-site array.

    status=0 success; 1 nonfinite integration; 2 subdivision resource limit.
    native_guard_exceeded flags |P|>10 without clipping or discarding the result.
    """
    arr=np.asarray(fields_ec,dtype=np.float64)
    if arr.ndim==1:
        arr=arr[None,:]
    if arr.ndim!=2 or arr.shape[1] not in (900,950):
        raise ValueError("Expected (batch,900) or (batch,950) native E/Ec values")
    if not np.isfinite(arr).all():
        raise ValueError("All supplied fields must be finite")
    if arr.shape[1]==900:
        arr=np.concatenate((arr,np.repeat(arr[:,-1:],50,axis=1)),axis=1)
    arr=np.ascontiguousarray(arr)
    if int(fidelity)!=fidelity or fidelity<1:
        raise ValueError("fidelity must be an integer >=1")
    if int(max_substeps)!=max_substeps or max_substeps<1:
        raise ValueError("max_substeps must be an integer >=1")
    if method not in ("rk4","legacy_euler"):
        raise ValueError("Unknown integration method")
    method_id=int(method=="rk4")
    if initial_p is None:
        initial=np.zeros((2,N,N),dtype=np.float64)
    else:
        initial=np.asarray(initial_p,dtype=np.float64)
        if initial.shape[-1]==2 and initial.shape[0]!=2:
            initial=initial.transpose(2,0,1)
        if initial.ndim!=3 or initial.shape[0]!=2 or initial.shape[1]!=initial.shape[2]:
            raise ValueError("Invalid initial polarization shape")
        initial=np.ascontiguousarray(initial)
    n=initial.shape[1]
    defects=default_defects() if defects_ec is None else np.asarray(defects_ec,dtype=np.float64).reshape(n,n,2)
    if defects.shape!=(n,n,2):
        raise ValueError("Defect grid must match initial polarization grid")
    if not np.isfinite(initial).all() or not np.isfinite(defects).all():
        raise ValueError("Initial polarization and defects must be finite")
    defects=np.ascontiguousarray(defects*EC)
    if return_history:
        out=[_integrate(x,defects,initial,int(fidelity),method_id,True,float(coupling),float(dep_alpha),int(max_substeps)) for x in arr]
        r,j,p,m,g,s,nrhs,nsub,hist=map(np.asarray,zip(*out))
    else:
        r,j,p,m,g,s,nrhs,nsub=_batch(arr,defects,initial,int(fidelity),method_id,float(coupling),float(dep_alpha),int(max_substeps))
    result={"reward":r,"curl":r,"interior":j,"final_P":p,"max_abs_P":m,
            "native_guard_exceeded":g,"status":s,"valid":(s==0)&np.isfinite(r),
            "rhs_calls":nrhs,"substeps":nsub,"field_peak_ec":np.max(np.abs(arr),axis=1),
            "fidelity":int(fidelity),"method":method}
    if return_history:
        result["P_history"]=hist
        result["mean_P"]=hist.mean(axis=(3,4))
        ch=np.empty((arr.shape[0],arr.shape[1]))
        ih=np.empty_like(ch)
        for b in range(arr.shape[0]):
            for t in range(arr.shape[1]):
                ch[b,t],ih[b,t],_=_curl(hist[b,:,t])
        result["curl_history"]=ch
        result["interior_history"]=ih
        result["time"]=TIME.copy()
        result["field_ec"]=arr
    return result


def curl_field(p):
    """Native periodic centered curl diagnostic for P shaped (2,n,n)."""
    return _curl(np.asarray(p,dtype=np.float64))[2]


In [ ]:
%%writefile FerroSim_Grammar_runtime/dkl.py
"""Nonlinear neural feature kernel + exact GP trained on fixed waveform samples."""
import numpy as np
import torch
from torch import nn
class DKL(nn.Module):
    def __init__(self,seed=0):
        super().__init__();torch.manual_seed(seed)
        self.net=nn.Sequential(nn.Linear(65,32),nn.Tanh(),nn.Linear(32,8)).double()
        self.logl=nn.Parameter(torch.tensor(0.,dtype=torch.double));self.logsig=nn.Parameter(torch.tensor(0.,dtype=torch.double))
        self.lognoise=nn.Parameter(torch.tensor(-2.,dtype=torch.double));self.optim=torch.optim.Adam(self.parameters(),lr=.015)
        self.cache=None
    def features(self,w):
        # 64 sampled waveform values plus segment count
        return self.net(torch.asinh(w))
    def kernel(self,a,b):
        dist=torch.cdist(a,b).square().clamp_min(0);return torch.exp(2*self.logsig)*torch.exp(-dist/(2*torch.exp(2*self.logl)))
    def fit(self,X,y,steps=35):
        from representation import fields,complexity
        w=fields(X,64);k=complexity(X)['segments'].reshape(-1,1)
        z=np.c_[w/30.,k/16.];self.ycenter=float(np.mean(y));self.yscale=max(float(np.std(y)),1e-3)
        self.X=torch.tensor(z,dtype=torch.double);self.y=torch.tensor((np.asarray(y)-self.ycenter)/self.yscale,dtype=torch.double)
        self.train()
        for _ in range(steps):
            self.optim.zero_grad();f=self.features(self.X);K=self.kernel(f,f);noise=torch.exp(2*self.lognoise)+1e-5
            L=torch.linalg.cholesky(K+noise*torch.eye(len(X),dtype=torch.double))
            alpha=torch.cholesky_solve(self.y[:,None],L)
            loss=(.5*(self.y[:,None]*alpha).sum()+torch.log(torch.diag(L)).sum())/len(X)
            loss.backward();self.optim.step()
            with torch.no_grad():self.logl.clamp_(-4,4);self.logsig.clamp_(-4,4);self.lognoise.clamp_(-6,1)
        self.eval()
        with torch.no_grad():
            self.f=self.features(self.X);K=self.kernel(self.f,self.f);self.L=torch.linalg.cholesky(K+(torch.exp(2*self.lognoise)+1e-5)*torch.eye(len(X)))
            self.alpha=torch.cholesky_solve(self.y[:,None],self.L)
        return self
    def predict(self,X):
        from representation import fields,complexity
        if not len(X):return np.array([]),np.array([])
        w=fields(X,64);k=complexity(X)['segments'].reshape(-1,1)
        with torch.no_grad():
            x=torch.tensor(np.c_[w/30.,k/16.],dtype=torch.double);f=self.features(x);c=self.kernel(f,self.f)
            mean=(c@self.alpha).squeeze();sol=torch.linalg.solve_triangular(self.L,c.T,upper=False)
            std=(torch.exp(2*self.logsig)-sol.square().sum(0)).clamp_min(1e-9).sqrt()
            return np.atleast_1d(mean.numpy()*self.yscale+self.ycenter),np.atleast_1d(std.numpy()*self.yscale)


In [ ]:
%%writefile FerroSim_Grammar_runtime/experiment.py
"""Grammar-based FerroSim optimization; all rows correspond to actual physical simulator calls."""
import os,time,argparse,json
os.environ.setdefault('NUMBA_NUM_THREADS','1');os.environ.setdefault('OMP_NUM_THREADS','1');os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
from pathlib import Path
import numpy as np,pandas as pd
from representation import OPS,decode,fields,initial_design,sample_one,mutate,complexity
from oracle import evaluate_fields,default_defects
from dkl import DKL
METHODS=['GA-DKL','QD-DKL']
GRAMMARS=['simple','full']

def disorder_maps(folder,n=20):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    ref=default_defects();active=ref.reshape(-1,2);active=active[np.any(active!=0,axis=1)]
    assert len(active)==19
    out={'reference':ref};np.save(folder/'reference.npy',ref)
    for k in range(n-1):
        rng=np.random.default_rng(20261122+k);pos=rng.choice(400,19,replace=False);m=np.zeros((400,2));m[pos]=active[rng.permutation(19)]
        out[f'd{k:02d}']=m.reshape(20,20,2);np.save(folder/f'd{k:02d}.npy',out[f'd{k:02d}'])
    return out

def propose(method,grammar,X,y,rng,count):
    ix=np.argsort(y)[::-1]
    pool=X[ix[:min(32,len(ix))]]
    if method.startswith('QD'):
        comp=complexity(X);labels=[(int(comp['segments'][i]),int(comp['n_ringdowns'][i]+comp['n_bursts'][i])) for i in range(len(X))]
        archive={}
        for i,key in enumerate(labels):
            if key not in archive or y[i]>y[archive[key]]:archive[key]=i
        pool=X[list(archive.values())]
    z=[];ops=[]
    for _ in range(count):
        if rng.random()<.10:
            child=sample_one(rng,grammar);label='restart'
        else:
            parent=pool[int(rng.integers(len(pool)))];child,label=mutate(parent,rng,grammar)
            if rng.random()<.20:child,_=mutate(child,rng,grammar)
        z.append(child);ops.append(label)
    return np.array(z),ops

def eval_controls(z,d,oracle_method='rk4',fidelity=1):
    v=evaluate_fields(fields(z,900),defects_ec=d,method=oracle_method,fidelity=fidelity,max_substeps=100000)
    reward=np.asarray(v['reward']);valid=np.asarray(v['valid']) & ~np.asarray(v['native_guard_exceeded']) & np.isfinite(reward)
    return np.where(valid,reward,np.nan),valid,np.asarray(v['interior']),np.asarray(v['rhs_calls'])

def one_run(mp,defects,method,grammar,seed,out,initial=64,generations=32,batch=32,candidates=256,steps=35,oracle_method='rk4'):
    folder=out/mp/(grammar+'__'+method)/f'seed_{seed}';folder.mkdir(parents=True,exist_ok=True)
    history=folder/'history.csv';serialized=folder/'programs.npz';expected=initial+generations*batch
    if history.exists() and serialized.exists():
        prior=pd.read_csv(history)
        if len(prior)==expected:
            return prior
    rng=np.random.default_rng(9011+seed*10007+int(mp[1:]) *101 if mp.startswith('d') else 9011+seed*10007)
    X=initial_design(seed,initial,grammar);Y=[];rows=[]
    for gen in range(generations+1):
        if gen==0:
            candidates_selected=X;ops=['initial']*len(X);mean=np.full(len(X),np.nan);std=mean.copy()
        else:
            yy=np.asarray(Y);good=np.isfinite(yy)
            pool,opnames=propose(method,grammar,X[good],yy[good],rng,candidates)
            if good.sum()>=2:
                model=DKL(seed+gen);model.fit(X[good],yy[good],steps=steps)
                mu,sig=model.predict(pool)
            else:mu=np.zeros(len(pool));sig=np.ones(len(pool))
            choice=np.argsort(-(mu+2*sig))[:batch]
            candidates_selected=pool[choice];ops=[opnames[int(i)] for i in choice];mean=mu[choice];std=sig[choice]
            X=np.vstack([X,candidates_selected])
        reward,valid,interior,rhs=eval_controls(candidates_selected,defects,oracle_method)
        Y.extend(reward.tolist());comp=complexity(candidates_selected)
        for j,z in enumerate(candidates_selected):
            seq=decode(z);rows.append(dict(map=mp,grammar=grammar,method=method,seed=seed,evaluation=len(rows)+1,generation=gen,reward=float(reward[j]),valid=bool(valid[j]),interior=float(interior[j]),rhs_calls=int(rhs[j]),operations=int(comp['segments'][j]),effective_operations=int(comp['effective_segments'][j]),turns=int(comp['turns'][j]),peak_ec=float(comp['peak_ec'][j]),rms_ec=float(comp['rms_ec'][j]),param_count=int(comp['parameter_count'][j]),n_ringdowns=int(comp['n_ringdowns'][j]),n_bursts=int(comp['n_bursts'][j]),n_pulses=int(comp['n_pulses'][j]),operator=ops[j],predicted_mean=float(mean[j]),predicted_std=float(std[j]),program=' > '.join(OPS[int(o)] for o in seq[:,0])))
        pd.DataFrame(rows).to_csv(history,index=False);np.savez_compressed(serialized,programs=X)
    return pd.DataFrame(rows)

def run(out,maps,seeds=range(3),initial=64,generations=32,batch=32,candidates=256,steps=35,oracle_method='rk4'):
    out=Path(out);out.mkdir(parents=True,exist_ok=True);all=[];start=time.time()
    for name,d in maps.items():
        for seed in seeds:
            for grammar in GRAMMARS:
                for method in METHODS:
                    print(f'{time.time()-start:.1f}s map={name} seed={seed} grammar={grammar} method={method}',flush=True)
                    all.append(one_run(name,d,method,grammar,seed,out,initial,generations,batch,candidates,steps,oracle_method))
    data=pd.concat(all,ignore_index=True);data.to_csv(out/'all_evaluations.csv',index=False)
    print('total measured:',len(data),'valid:',int(data.valid.sum()),'elapsed:',round(time.time()-start,1),flush=True)
    return data

if __name__=='__main__':
    ap=argparse.ArgumentParser();ap.add_argument('--out',default='Grammar_results');ap.add_argument('--pilot',action='store_true');ap.add_argument('--maps',type=int,default=20);ap.add_argument('--seeds',type=int,default=3);ap.add_argument('--initial',type=int,default=64);ap.add_argument('--generations',type=int,default=32);ap.add_argument('--batch',type=int,default=32);ap.add_argument('--candidates',type=int,default=256);ap.add_argument('--steps',type=int,default=35);ap.add_argument('--oracle-method',choices=['rk4','legacy_euler'],default='rk4');args=ap.parse_args()
    if args.pilot:args.maps=1;args.seeds=1;args.initial=8;args.generations=1;args.batch=4;args.candidates=16;args.steps=6;args.oracle_method='legacy_euler'
    out=Path(args.out);maps=disorder_maps(out/'disorder_maps',args.maps)
    run(out,maps,range(args.seeds),args.initial,args.generations,args.batch,args.candidates,args.steps,args.oracle_method)


In [ ]:
%%writefile FerroSim_Grammar_runtime/analysis.py
"""Analyzes strictly measured grammar-optimizer evaluations, never synthetic performance data."""
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from representation import decode,fields,OPS,complexity
from oracle import evaluate_fields

def load(folder):
    f=Path(folder)/'all_evaluations.csv'
    if not f.exists():raise FileNotFoundError(f'Run experiment first: {f}')
    return pd.read_csv(f)

def get_program(folder,row):
    f=Path(folder)/str(row['map'])/(str(row['grammar'])+'__'+str(row['method']))/f"seed_{int(row['seed'])}"/'programs.npz'
    with np.load(f) as data:return data['programs'][int(row['evaluation'])-1].copy()

def summary(df):
    out=[]
    for key,g in df.groupby(['map','grammar','method','seed']):
        g=g[g.valid & np.isfinite(g.reward)]
        if len(g):
            b=g.loc[g.reward.idxmax()];out.append(dict(zip(['map','grammar','method','seed'],key))|{'best_reward':b.reward,'operations':b.operations,'turns':b.turns,'evaluation_at_best':b.evaluation,'program':b.program,'ringdowns':b.n_ringdowns,'bursts':b.n_bursts})
    return pd.DataFrame(out)

def plot_convergence(df):
    fig,ax=plt.subplots(figsize=(10,5))
    for (grammar,method),g in df[df['map']=='reference'].groupby(['grammar','method']):
        curves=[]
        for seed,z in g.groupby('seed'):
            y=z.sort_values('evaluation').reward.ffill().fillna(-np.inf).to_numpy();curves.append(np.maximum.accumulate(y))
        x=np.arange(1,len(curves[0])+1);c=np.array(curves)
        ax.plot(x,c.mean(0),label=f'{grammar}/{method}')
        if len(c)>1:ax.fill_between(x,c.mean(0)-c.std(0),c.mean(0)+c.std(0),alpha=.15)
    ax.set(xlabel='Actual physical evaluations per run',ylabel='Best observed curl',title='Grammar optimization convergence');ax.legend(fontsize=8);ax.grid(alpha=.2);fig.tight_layout();return fig

def plot_champions(df,folder):
    fig,axs=plt.subplots(2,2,figsize=(13,7),sharex=True)
    for ax,((grammar,method),g) in zip(axs.flat,df[(df['map']=='reference')&df.valid].groupby(['grammar','method'])):
        b=g.loc[g.reward.idxmax()];z=get_program(folder,b);t=np.linspace(0,1,900)
        ax.plot(t,fields(z,900)[0],lw=1.6)
        ax.set(title=f'{grammar} / {method}: R={b.reward:.4f}, ops={b.operations}',ylabel='E/Ec',xlabel='Normalized time');ax.grid(alpha=.2)
        ax.text(.02,.02,b.program,transform=ax.transAxes,fontsize=7,va='bottom',wrap=True)
    fig.tight_layout();return fig

def plot_pareto(df):
    fig,ax=plt.subplots(figsize=(9,5))
    for (grammar,method),g in df[(df['map']=='reference')&df.valid].groupby(['grammar','method']):
        best=-np.inf;xx=[];yy=[]
        for k in sorted(g.operations.unique()):
            v=g.loc[g.operations==k,'reward'].max()
            if v>best:best=v;xx.append(k);yy.append(v)
        ax.step(xx,yy,where='post',marker='o',label=f'{grammar}/{method}')
    ax.set(xlabel='Number of primitive operations',ylabel='Max curl',title='Measured reward–program complexity Pareto');ax.grid(alpha=.2);ax.legend(fontsize=8);fig.tight_layout();return fig

def plot_evolution(df,folder):
    fig,axes=plt.subplots(2,2,figsize=(12,8),layout='constrained')
    for ax,((grammar,method),g) in zip(axes.flat,df[(df['map']=='reference')&(df.seed==0)].groupby(['grammar','method'])):
        g=g.sort_values('evaluation');inc=-np.inf;waves=[]
        for _,r in g.iterrows():
            if r.valid and np.isfinite(r.reward) and r.reward>inc:inc=r.reward;champ=get_program(folder,r)
            elif not waves:champ=get_program(folder,r)
            waves.append(fields(champ,256)[0]);
        im=ax.imshow(waves,origin='lower',aspect='auto',extent=[0,1,1,len(waves)],interpolation='nearest')
        ax.set(xlabel='Normalized time',ylabel='Evaluation',title=f'{grammar}/{method} incumbent E(t)');fig.colorbar(im,ax=ax,shrink=.75)
    return fig

def plot_disorder(df):
    summ=summary(df);fig,axes=plt.subplots(1,2,figsize=(12,4))
    for (grammar,method),g in summ.groupby(['grammar','method']):
        means=g.groupby('map').best_reward.mean().sort_index()
        axes[0].plot(range(len(means)),means.values,label=f'{grammar}/{method}',marker='.',lw=1)
        axes[1].scatter(g.operations,g.best_reward,label=f'{grammar}/{method}',alpha=.5,s=14)
    axes[0].set(title='Mean reward across disorder maps',xlabel='Disorder index',ylabel='Best reward')
    axes[1].set(title='Champion complexity across disorder',xlabel='Operations',ylabel='Best reward')
    for ax in axes:ax.grid(alpha=.2);ax.legend(fontsize=7)
    fig.tight_layout();return fig

def variance(df):
    summ=summary(df);rows=[]
    for (grammar,method),g in summ.groupby(['grammar','method']):
        means=g.groupby('map').best_reward.mean();within=g.groupby('map').best_reward.var(ddof=1)
        rows.append(dict(grammar=grammar,method=method,across_disorder=float(means.var(ddof=1)) if len(means)>1 else np.nan,within_disorder_across_seeds=float(within.mean()) if within.notna().any() else np.nan))
    return pd.DataFrame(rows)

def time_to_target(df,fraction=.95):
    rows=[]
    for mp,g in df.groupby('map'):
        valid=g[g.valid & np.isfinite(g.reward)];
        if not len(valid):continue
        baseline=valid[valid.generation==0].reward.min();goal=baseline+fraction*(valid.reward.max()-baseline)
        for (grammar,method,seed),q in g.groupby(['grammar','method','seed']):
            hits=q[q.valid & (q.reward>=goal)]
            rows.append(dict(map=mp,grammar=grammar,method=method,seed=seed,threshold=goal,fraction=fraction,first_evaluation=float(hits.evaluation.min()) if len(hits) else np.nan))
    return pd.DataFrame(rows)

def transfer(df,folder,maps,oracle_method='rk4',fidelity=1):
    rows=[];summ=summary(df)
    for _,r in summ.iterrows():
        q=df[(df['map']==r['map'])&(df.grammar==r.grammar)&(df.method==r.method)&(df.seed==r.seed)]
        best=q.loc[q.reward.idxmax()];z=get_program(folder,best);wave=fields(z,900)
        for target,d in maps.items():
            v=evaluate_fields(wave,defects_ec=d,method=oracle_method,fidelity=fidelity)
            rows.append(dict(source=r['map'],target=target,grammar=r.grammar,method=r.method,seed=int(r.seed),reward=float(v['reward'][0]),valid=bool(v['valid'][0] and not v['native_guard_exceeded'][0])))
    return pd.DataFrame(rows)

def plot_transfer(tr):
    fig,axes=plt.subplots(2,2,figsize=(12,10),layout='constrained')
    for ax,((grammar,method),g) in zip(axes.flat,tr.groupby(['grammar','method'])):
        p=g.pivot_table(index='source',columns='target',values='reward',aggfunc='mean');im=ax.imshow(p,aspect='auto');fig.colorbar(im,ax=ax,shrink=.7)
        ax.set(title=f'{grammar}/{method}',xlabel='Target disorder',ylabel='Source disorder')
    return fig

def motif_frequency(df):
    records=[]
    for _,r in df[df.valid].iterrows():
        ops=r.program.split(' > ')
        for op in ops:records.append((r['grammar'],r.method,op,r.reward))
    return pd.DataFrame(records,columns=['grammar','method','operation','reward']).groupby(['grammar','method','operation']).agg(count=('reward','size'),mean_reward=('reward','mean')).reset_index()

def refinement(df,folder,maps,method='rk4',fidelity=2):
    rows=[]
    for _,b in summary(df).iterrows():
        if b['map'] not in maps:continue
        q=df[(df['map']==b['map'])&(df.grammar==b.grammar)&(df.method==b.method)&(df.seed==b.seed)]
        z=get_program(folder,q.loc[q.reward.idxmax()]);v=evaluate_fields(fields(z,900),defects_ec=maps[b['map']],method=method,fidelity=fidelity)
        rows.append(dict(map=b['map'],grammar=b.grammar,method=b.method,seed=b.seed,original=b.best_reward,refined=float(v['reward'][0]),valid=bool(v['valid'][0])))
    return pd.DataFrame(rows)

def export(df,folder):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True);d=folder/'analysis';d.mkdir(exist_ok=True)
    summary(df).to_csv(d/'run_summary.csv',index=False)
    variance(df).to_csv(d/'variance.csv',index=False)
    time_to_target(df,.95).to_csv(d/'target_95.csv',index=False)
    time_to_target(df,.99).to_csv(d/'target_99.csv',index=False)
    motif_frequency(df).to_csv(d/'motif_frequencies.csv',index=False)
    for name,fn in [('convergence',lambda:plot_convergence(df)),('champions',lambda:plot_champions(df,folder)),('pareto',lambda:plot_pareto(df)),('evolution',lambda:plot_evolution(df,folder)),('disorder',lambda:plot_disorder(df))]:
        fig=fn();fig.savefig(d/(name+'.png'),dpi=150);plt.close(fig)
    return d


## 3. Embedded physical reference disorder: identical to earlier FerroSim notebooks

In [ ]:
REF_B85='c-oun7wQ`j$;eQ~P_3SlTAW;@Zl$1ZlV+i=qoAIaUsO_*m=~X4l#&V(cT3DEP6dh=XCxM+0{I#y1_n9`Mw&VbwF+bcE{0JsARsI7O?HNjtHWS%PcGN8!ix894#OpUE6&gKjJ-L0{6i|p`+h9lcB8k$XilRI$jDo~(<az&#D=fq!$%=rXIw_p*DwWLGy3~pKKB|n@%uK&eo@W_kJ0oxLcv9uMND&#51#gb(VCv40=%@VpVc*8TsMolj@b0hXZTd}(D~t$-+ia*3B8W-av06YqktT+b~F3hU00n))6XykMtK`gZQA5BY~uGoilo|;)}Yb!Its`FS69yYQe7TMv-I<ocLmp>uF>(8p#pejU*CT$cC`I83Qz$6MT1~F'
(Path(ROOT)/'defect_list_seed.npy').write_bytes(zlib.decompress(base64.b85decode(REF_B85)))
from representation import OPS,decode,encode,fields,initial_design,mutate,complexity,sample_one,compile_one
from experiment import disorder_maps,eval_controls
from analysis import load,summary,plot_convergence,plot_champions,plot_pareto,plot_evolution,plot_disorder,variance,time_to_target,transfer,plot_transfer,motif_frequency,refinement,export
print('Embedded reference defect map shape:',np.load(ROOT/'defect_list_seed.npy').shape)

## 4. Grammar compilation and physical admissibility checks

**Primitives:** `ramp` linearly changes the applied field to a target; `hold` retains current field; `reverse` ramps to the opposite polarity; `pulse` adds a smooth finite-width excursion; `ringdown` is an **actively applied damped oscillatory field** (`sin²` envelope times exponential and sinusoid, *not* passive sample relaxation); `burst` applies a smooth-envelope oscillation.

The opening and closing neutral segments are imposed by the compiler. Durations are positive and normalized to the fixed campaign length; program lengths are 2–12 active operations. The terminal field is guaranteed zero by the closing ramp and hold, and the initial field is zero by an initial hold. Interior operations are continuous in amplitude but not necessarily in slope. **No claim of hardware compliance** is made unless slew-rate, heating, and bandwidth constraints are separately enabled.

In [ ]:
rng=np.random.default_rng(2026)
examples={name:encode([[0,1,20,0,2,1],[i,1,0,25,3,.55],[0,1,0,0,2,1]]) for i,name in enumerate(OPS)}
fig,axs=plt.subplots(2,3,figsize=(13,6),layout='constrained')
for ax,(name,z) in zip(axs.flat,examples.items()):
    t=np.linspace(0,1,900);w=fields(z,900)[0];ax.plot(t,w);ax.set(title=name,xlabel='Normalized time',ylabel='Applied E/Ec');ax.grid(alpha=.2)
    assert np.max(np.abs(w[:40]))<1e-12 and np.max(np.abs(w[-23:]))<1e-10
    assert np.isfinite(w).all()
plt.show()
for grammar in ('simple','full'):
    z=sample_one(rng,grammar)
    for _ in range(200):
        z,operation=mutate(z,rng,grammar)
        s=decode(z);wave=fields(z,900)[0]
        assert 2<=len(s)<=12 and np.isfinite(wave).all()
        if grammar=='simple':assert np.max(s[:,0])<=2
print('Passed: six primitives, waveform continuity/boundaries, variable-length mutation, and grammar restrictions.')

## 5. Twenty disorder maps (shared randomization protocol with earlier studies)

In [ ]:
DISORDER_MAPS=disorder_maps(ROOT/'disorder_maps',20)
fig,axes=plt.subplots(4,5,figsize=(12,9),layout='constrained')
for ax,(label,d) in zip(axes.flat,DISORDER_MAPS.items()):
    ax.imshow(np.linalg.norm(d,axis=-1));ax.set_title(label);ax.set_xticks([]);ax.set_yticks([])
plt.show()

## 6. Physically evaluate the experimental design

- `PILOT=True`: reference disorder, one seed, **8 initial + 4 acquired evaluations** per method/grammar, **48 physical evaluations total**, *legacy Euler*, small DKL model. The JIT compilation and FerroSim evaluation can take substantial time.
- `PILOT=False`: 20 maps, three seeds, 64 initial + 32 × 32 acquisitions per run, **261,120 evaluations total**, **RK4**. This is not an interactive quick run and should be scheduled on sufficient compute.
- All methods select from an acquisition pool with posterior mean + 2 × uncertainty; GA uses elite selection, QD uses program-length and oscillatory-operation archive niches. Results are not interchanged with earlier studies until fidelity and other settings are matched.
- The `full` grammar can represent all `simple` programs; however, the two optimized search processes can have different search efficiencies.

In [ ]:
import subprocess
RESULTS=ROOT/('pilot_results' if PILOT else 'full_results')
if RUN_EXPERIMENT:
    cmd=[sys.executable,str(ROOT/'experiment.py'),'--out',str(RESULTS)]
    if PILOT:cmd+=['--pilot']
    else:cmd+=['--maps','20','--seeds','3','--initial','64','--generations','32','--batch','32','--candidates','256','--steps','35','--oracle-method','rk4']
    subprocess.run(cmd,check=True)
else:
    print('Physical evaluation skipped. To execute, set RUN_EXPERIMENT=True in §1 and run from the top.')

## 7. Read observed rewards; no synthetic or placeholder numerical results

In [ ]:
HAS_RESULTS=(RESULTS/'all_evaluations.csv').exists()
if HAS_RESULTS:
    df=load(RESULTS)
    display(summary(df).sort_values(['map','grammar','method','seed']).head(30))
    print('Actual physical evaluations:',len(df),'valid:',int(df.valid.sum()))
else:
    print('No measured results; analysis plots below are intentionally disabled until §6 completes.')

## 8. Learning curves, time-to-target, and run-level variability

In [ ]:
if HAS_RESULTS:
    display(time_to_target(df,.95).head(20));display(time_to_target(df,.99).head(20));display(variance(df))
    display(plot_convergence(df));plt.show()

## 9. Reward–operation complexity Pareto fronts and champions

In [ ]:
if HAS_RESULTS:
    display(plot_pareto(df));plt.show()
    display(plot_champions(df,RESULTS));plt.show()

## 10. Two-dimensional `imshow`: evolving incumbent waveforms

In [ ]:
if HAS_RESULTS:
    display(plot_evolution(df,RESULTS));plt.show()

## 11. Disorder-dependent rewards and operation grammar

In [ ]:
if HAS_RESULTS:
    display(plot_disorder(df));plt.show()
    display(motif_frequency(df));
    display(df.groupby(['grammar','method','operator']).agg(count=('reward','size'),mean_reward=('reward','mean')).reset_index())

## 12. Cross-disorder transfer matrix (optional, physically evaluated)

Champion from each source disorder evaluated unchanged on each target disorder. Compare transfer values against optimized target-disorder champions before interpreting normalization. Never infer transfer results from surrogate predictions.

In [ ]:
if HAS_RESULTS and RUN_TRANSFER:
    active={name:DISORDER_MAPS[name] for name in sorted(df['map'].unique())}
    tr=transfer(df,RESULTS,active,oracle_method=('legacy_euler' if PILOT else 'rk4'))
    tr.to_csv(RESULTS/'transfer.csv',index=False)
    display(tr.head(25));display(plot_transfer(tr));plt.show()
else:print('Transfer not run (requires evaluated records; toggle RUN_TRANSFER).')

## 13. Independent higher-fidelity validation (optional)

Reevaluate champion programs using subdivided RK4 fidelity 2. The difference from the original optimizer reward is a numerical diagnostic, not itself a proof of generalization.

In [ ]:
if HAS_RESULTS and RUN_REFINEMENT:
    val=refinement(df,RESULTS,DISORDER_MAPS,method='rk4',fidelity=2)
    val.to_csv(RESULTS/'refinement.csv',index=False);display(val)
else:print('No refinement requested.')

## 14. Export all numerical records, plots, and motifs

In [ ]:
if HAS_RESULTS:
    print('Exported actual evaluations and analysis to:',export(df,RESULTS))
else:print('Results unavailable; grammar construction and tests still ran independently.')

## 15. Interpretation, limitations, and follow-up experiment

1. **Primary test:** does the full operation grammar improve observed best curl or reach reward targets with fewer evaluations than ramp/hold/reverse-only control? Report uncertainty over disorder and seeds, not a one-run victory.
2. **Reward vs complexity:** compare operation-count Pareto fronts; inspect effective waveform complexity and amplitude/energy costs separately.
3. **Motif transfer:** identify recurring operation subsequences in successful protocols, then evaluate unchanged programs across disorders.
4. **Fidelity:** the quick pilot uses legacy Euler; never compare its raw curl to RK4 Fourier/Legendre values. Confirm conclusions with RK4 and higher-fidelity refinement.
5. **Operation semantics:** ringdown is an imposed field, not the natural relaxation of sample polarization. Passive relaxation requires a separate simulator state-evolution operator with a zero/hold field.
6. **Optimization fairness:** 2 methods × 2 grammars are implemented here. A fully trained GFlowNet or diffusion model is outside this experiment and should not be claimed as such.
7. **Constraints:** the current compiled waveform is continuous with neutral endpoints, but hardware slew-rate, bandwidth and heating bounds should be implemented and tracked before direct experimental use.

The notebook contains no precomputed full-study results. All performance plots are generated only from actual simulator measurements.